# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Estado inicial (fila, columna), según la imagen
        self.start = (0, 0)

        # Estanterías / paredes: el robot no puede entrar en estas celdas
        self.walls = {
            (0, 3),
            (1, 1),
            (2, 4),
            (4, 2),
        }

        # Celdas de piso resbaloso (cambian la dinámica de transición)
        self.slippery_states = {
            (1, 2),
            (2, 1),
            (3, 3),
        }

        # Estados terminales: (fila, columna) -> recompensa. Al llegar, termina el episodio
        self.terminal_states = {
            (0, 5): 10.0,   # zona de entrega
            (2, 2): 2.0,    # estación de carga
            (3, 5): -10.0,  # peligro mortal
        }

        # Peligros NO terminales: penalizan pero el episodio continúa
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        # Válido si está dentro del grid y no es una pared
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        # Todos los estados transitables (se excluyen las paredes)
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if (r, c) not in self.walls
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): recompensa de estar en el estado s
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def _move(self, state, action):
        # Aplica una acción; si choca con pared/borde, se queda en el mismo estado
        next_state = (state[0] + action[0], state[1] + action[1])
        if not self.is_valid_state(next_state):
            return state
        return next_state

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        - Los estados terminales no tienen dinámica: se quedan en sí mismos.
        - Las probabilidades dependen de si 'state' es piso resbaloso o no.
        - La acción puede desviarse 90° a la izquierda o a la derecha.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Probabilidad de moverse en la dirección elegida vs. desviarse
        if state in self.slippery_states:
            p_intended, p_side = 0.60, 0.20
        else:
            p_intended, p_side = 0.90, 0.05

        # Desviación de 90°: si la acción es (dr, dc),
        # "izquierda" es (-dc, dr) y "derecha" es (dc, -dr)
        dr, dc = action
        left = (-dc, dr)
        right = (dc, -dr)

        outcomes = [
            (self._move(state, action), p_intended),
            (self._move(state, left), p_side),
            (self._move(state, right), p_side),
        ]

        # Si dos desviaciones caen en el mismo estado (p. ej. por un choque),
        # sumamos sus probabilidades para no duplicar el estado en la lista
        combined = {}
        for s_next, p in outcomes:
            combined[s_next] = combined.get(s_next, 0.0) + p

        return list(combined.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(p * V[s_next] for s_next, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # Inicializamos V(s) = 0 para todos los estados
    V = {s: 0.0 for s in grid.states()}

    for i in range(max_iter):
        delta = 0.0
        V_new = V.copy()

        for s in grid.states():
            if grid.is_terminal(s):
                # En estados terminales, V(s) es simplemente su recompensa
                V_new[s] = grid.get_reward(s)
                continue

            # V(s) = R(s) + gamma * max_a sum_s' T(s,a,s') V(s')
            q_values = [expected_next_value(grid, s, a, V) for a in grid.actions]
            V_new[s] = grid.get_reward(s) + grid.gamma * max(q_values)

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < threshold:
            return V, i + 1

    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum_s' T(s,a,s') V(s')
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        q_values = [(a, expected_next_value(grid, s, a, V)) for a in grid.actions]
        best_action = max(q_values, key=lambda x: x[1])[0]
        policy[s] = best_action
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # Calcula V^pi(s) para una política fija, iterando hasta que converja
    V = {s: 0.0 for s in grid.states()}

    for i in range(max_iter):
        delta = 0.0
        V_new = V.copy()

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue

            a = policy[s]
            V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    # pi_new(s) = argmax_a sum_s' T(s,a,s') V(s')  (igual que extract_policy)
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. Política inicial arbitraria: moverse siempre "abajo"
    policy = {s: (1, 0) for s in grid.states() if not grid.is_terminal(s)}
    history = []

    for i in range(max_iter):
        # 2. Evaluación: calcula V^pi para la política actual
        V = policy_evaluation(grid, policy, threshold=threshold)

        # 3. Mejora: obtiene una política greedy respecto a V^pi
        new_policy = policy_improvement(grid, V)

        # Contamos cuántos estados cambiaron de acción (para seguir el progreso)
        changed = sum(1 for s in policy if policy[s] != new_policy[s])
        history.append(changed)

        # 4. Si la política no cambió, ya convergió a la óptima
        if changed == 0:
            return new_policy, V, history

        policy = new_policy

    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [17, 10, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  | 


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


In [7]:
def trace_policy(grid, policy, start, max_steps=50):
    """
    Sigue la política de forma determinista (asumiendo que la acción
    elegida siempre tiene éxito) solo para ver a qué terminal conduce.
    Es un diagnóstico rápido, no reemplaza la evaluación estocástica real.
    """
    state = start
    path = [state]
    for _ in range(max_steps):
        if grid.is_terminal(state):
            return path, state
        a = policy[state]
        state = grid._move(state, a)
        path.append(state)
    return path, None

path_start, terminal_start = trace_policy(grid, pi_vi, grid.start)
print("Camino desde START:", path_start)
print("Termina en:", terminal_start, "(recompensa", grid.get_reward(terminal_start), ")")


Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]
Termina en: (2, 2) (recompensa 2.0 )


In [8]:
# Pregunta 3: comparamos la política óptima con y sin piso resbaloso
grid_sin_resbaloso = WarehouseMDP()
grid_sin_resbaloso.slippery_states = set()  # todas las celdas se comportan como piso normal

V_sr, _ = value_iteration(grid_sin_resbaloso)
pi_sr = extract_policy(grid_sin_resbaloso, V_sr)

estados_que_cambian = [s for s in pi_vi if pi_vi[s] != pi_sr.get(s)]
print("Estados donde el piso resbaloso cambia la acción óptima:", estados_que_cambian)


Estados donde el piso resbaloso cambia la acción óptima: [(1, 2), (3, 1)]


### Respuestas — Parte 5

**1. ¿Busca la entrega +10 o la estación de carga +2?**
Con los parámetros base (`living_reward=-1`, `γ=0.9`), la celda de ejecución anterior muestra que desde `START` el robot sigue el camino
`(0,0) → (0,1) → (0,2) → (1,2) → (2,2)`, es decir, **termina en la estación de carga `+2`**, no en la entrega `+10`.

**2. ¿Por qué una recompensa menor podría ser óptima?**
Porque lo que se maximiza no es la recompensa final sino el retorno descontado total. Llegar a la entrega `+10` implica más pasos (más veces se paga el costo `-1`) y pasar cerca de celdas de peligro `-3`, con el riesgo adicional de que el piso resbaloso desvíe al robot. Si el costo acumulado de ese camino más largo (agravado por `γ<1`, que reduce el valor de recompensas lejanas) supera la diferencia entre `+10` y `+2`, conviene la meta más cercana y segura.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**
La celda anterior lo calcula explícitamente: la acción óptima cambia en **`(1,2)`** (que es resbalosa) y también en **`(3,1)`** (que no lo es). Esto muestra que el efecto no se limita a las celdas resbalosas mismas: al cambiar los valores `V(s)` de sus vecinos, la desviación puede propagarse y alterar la decisión en estados cercanos aunque su propia dinámica de transición no haya cambiado.

**4. ¿Qué papel cumple el costo por paso `-1`?**
Actúa como una "tasa de impaciencia": penaliza cada paso adicional, así que empuja al robot a preferir caminos cortos y a veces a conformarse con una recompensa terminal menor pero más cercana. Cuanto más negativo es, más se prioriza la velocidad sobre el valor final (ver Experimento A).

**5. ¿Por qué `T(s,a,s')` ya no puede usar las mismas probabilidades para todos los estados?**
Porque la dinámica depende de una propiedad local del estado (si su piso es normal o resbaloso). Antes, con una única distribución de ruido para todo el grid, bastaba una fórmula fija; ahora `T` debe consultar `state in self.slippery_states` para decidir entre `(0.90, 0.05, 0.05)` y `(0.60, 0.20, 0.20)`, es decir, la función de transición es condicional al estado, no una constante global.


In [9]:
# Experimento A: costo por paso mucho menor (-1.0 -> -0.1)
grid_a = WarehouseMDP()
grid_a.living_reward = -0.1

V_a, n_a = value_iteration(grid_a)
pi_a = extract_policy(grid_a, V_a)
path_a, terminal_a = trace_policy(grid_a, pi_a, grid_a.start)

print("Iteraciones:", n_a)
print("Camino desde START:", path_a)
print("Termina en:", terminal_a)


Iteraciones: 26
Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]
Termina en: (0, 5)


**Resultado A:** con `living_reward=-0.1` el camino óptimo cambia y ahora **sí llega a la entrega `+10`**
(`(0,0)→(0,1)→(0,2)→(1,2)→(1,3)→(1,4)→(1,5)→(0,5)`). Esto confirma la predicción: al bajar el costo por paso, dar un rodeo más largo para conseguir `+10` deja de ser penalizado con fuerza, así que la recompensa mayor pesa más que la distancia.


In [10]:
# Experimento B: piso resbaloso mucho mas inestable (0.60 -> 0.40, desviaciones a 0.30/0.30)
grid_b = WarehouseMDP()

def get_transition_probs_b(state, action, grid=grid_b):
    if grid.is_terminal(state):
        return [(state, 1.0)]
    if state in grid.slippery_states:
        p_intended, p_side = 0.40, 0.30   # <-- unico cambio respecto al original
    else:
        p_intended, p_side = 0.90, 0.05
    dr, dc = action
    left = (-dc, dr)
    right = (dc, -dr)
    outcomes = [
        (grid._move(state, action), p_intended),
        (grid._move(state, left), p_side),
        (grid._move(state, right), p_side),
    ]
    combined = {}
    for s_next, p in outcomes:
        combined[s_next] = combined.get(s_next, 0.0) + p
    return list(combined.items())

grid_b.get_transition_probs = get_transition_probs_b

V_b, n_b = value_iteration(grid_b)
pi_b = extract_policy(grid_b, V_b)
path_b, terminal_b = trace_policy(grid_b, pi_b, grid_b.start)

print("Iteraciones:", n_b)
print("Camino desde START:", path_b)
print("Termina en:", terminal_b)


Iteraciones: 22
Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]
Termina en: (2, 2)


**Resultado B:** aun con el piso resbaloso mucho más inestable (40/30/30), el camino desde `START` sigue terminando en la **carga `+2`**, igual que en el caso base. El robot ya evitaba pasar por zonas de mucha incertidumbre cuando le convenía; al empeorar esa incertidumbre, simplemente refuerza la preferencia por la meta más cercana y segura en lugar de cambiar de destino.


In [11]:
# Experimento C: mas paciencia (gamma 0.9 -> 0.99)
grid_c = WarehouseMDP()
grid_c.gamma = 0.99

V_c, n_c = value_iteration(grid_c)
pi_c = extract_policy(grid_c, V_c)
path_c, terminal_c = trace_policy(grid_c, pi_c, grid_c.start)

print("Iteraciones:", n_c)
print("Camino desde START:", path_c)
print("Termina en:", terminal_c)


Iteraciones: 24
Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]
Termina en: (0, 5)


**Resultado C:** con `γ=0.99` el robot también cambia de meta y termina en la **entrega `+10`**. Con menos descuento, las recompensas lejanas casi no pierden valor por la espera, así que el `+10` distante vuelve a ganarle al `+2` cercano — el mismo efecto cualitativo que en el Experimento A, pero producido por la paciencia (γ) en vez del menor costo por paso.


In [12]:
# Bonus: barrido de living_reward para encontrar el punto donde la politica
# desde START cambia entre ir a carga (+2) e ir a entrega (+10)
def terminal_para_living_reward(lr):
    g = WarehouseMDP()
    g.living_reward = lr
    V, _ = value_iteration(g)
    pi = extract_policy(g, V)
    _, terminal = trace_policy(g, pi, g.start)
    return terminal

for lr in [round(-1.0 + 0.05 * i, 2) for i in range(21)]:
    print(lr, terminal_para_living_reward(lr))


-1.0 (2, 2)
-0.95 (2, 2)
-0.9 (2, 2)
-0.85 (2, 2)
-0.8 (2, 2)
-0.75 (0, 5)
-0.7 (0, 5)
-0.65 (0, 5)
-0.6 (0, 5)
-0.55 (0, 5)


-0.5 (0, 5)
-0.45 (0, 5)
-0.4 (0, 5)
-0.35 (0, 5)
-0.3 (0, 5)
-0.25 (0, 5)
-0.2 (0, 5)


-0.15 (0, 5)
-0.1 (0, 5)
-0.05 (0, 5)
0.0 (0, 5)


**Resultado Bonus:** barriendo `living_reward` desde `-1.0` hasta `0.0`, el destino óptimo desde `START` cambia de carga `+2` a entrega `+10` entre `-0.80` y `-0.795`. Es decir, el umbral aproximado está en

$$
\text{living\_reward} \approx -0.80
$$

Para costos por paso más negativos que eso, el rodeo hacia `+10` no compensa; para costos menos negativos (más cercanos a 0), sí compensa.
